# Day 08 — Mock Interview Practice

**Week 2 · Day 8 of daily practice — INTERMEDIATE → ADVANCED level**

Today we step up from Week 1 fundamentals. Every question is a **multi-step problem or a real-world scenario** — the kind asked in mid- to senior-level product-company interviews. Expect to combine 2-3 techniques per question.

**18 questions · 8 categories:**

| # | Category | Questions | Difficulty |
|---|----------|-----------|------------|
| 1 | SQL Advanced | Q1-Q2 | [INTERMEDIATE] → [ADVANCED]
| 2 | Python — Loops/Functions/Lists/Dicts | Q3-Q4 | [INTERMEDIATE] → [ADVANCED]
| 3 | PySpark Coding | Q5-Q6 | [INTERMEDIATE] → [ADVANCED]
| 4 | Spark Optimization & Deep Spark | Q7-Q8 | [ADVANCED]
| 5 | ETL — Failures, Late Data, Fault Tolerance | Q9-Q11 | [INTERMEDIATE] → [ADVANCED]
| 6 | Data Modeling | Q12-Q13 | [ADVANCED]
| 7 | Snowflake | Q14-Q15 | [INTERMEDIATE] → [ADVANCED]
| 8 | Monitoring, Logging & Orchestration | Q16-Q18 | [ADVANCED]

**How to practice:** Read the problem, attempt it yourself for 15-20 minutes before opening the solution. Say your answer out loud as if explaining to an interviewer — interviewers grade communication as much as code.

In [0]:
# Setup — scratch schema for this notebook's Delta demos (idempotent)
spark.sql("CREATE SCHEMA IF NOT EXISTS day08_demo")
print("Setup done: day08_demo schema ready.")

## Q1 [INTERMEDIATE] — Gaps-and-Islands: User Sessionization

You are given an event log of user activity (user_id, event_ts). **A session is a sequence of events by the same user where consecutive events are at most 30 minutes apart.** If the gap between two consecutive events exceeds 30 minutes, a new session starts.

Write a query that assigns a `session_id` (sequential per user, ordered by time) to every event, and returns each user's sessions with: session start, session end, number of events, and session duration in minutes.

**Interview points:** This is the classic *gaps-and-islands* pattern — identify "islands" of consecutive rows and separate them at "gaps". The canonical trick is `row_number() minus a time-based grouping`, or a gap flag + cumulative sum. Be ready to explain both.

In [0]:
# Solution Q1 — Gaps-and-islands sessionization
# Islands = consecutive events within 30 min; gaps break the island.
from pyspark.sql import functions as F, Window

session_events = [
    ("u1", "2026-01-01 10:00:00"), ("u1", "2026-01-01 10:10:00"),  # session 1
    ("u1", "2026-01-01 10:25:00"),                                # still session 1 (15-min gap)
    ("u1", "2026-01-01 11:30:00"), ("u1", "2026-01-01 11:40:00"),  # session 2 (>30-min gap)
    ("u2", "2026-01-01 09:00:00"), ("u2", "2026-01-01 09:20:00"),  # u2 session 1
    ("u2", "2026-01-01 12:00:00"),                                # u2 session 2
]
df = spark.createDataFrame(session_events, ["user_id", "event_ts"]) \
    .withColumn("event_ts", F.to_timestamp("event_ts"))
df.createOrReplaceTempView("events")

result = spark.sql("""
WITH flagged AS (
  SELECT user_id, event_ts,
         CASE
           WHEN event_ts - LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts) > INTERVAL 30 MINUTES
                OR LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts) IS NULL
           THEN 1 ELSE 0
         END AS is_new_session
  FROM events
),
sessions AS (
  SELECT *,
         SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts
                                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_seq
  FROM flagged
)
SELECT user_id,
       CONCAT(user_id, '-', LPAD(CAST(session_seq AS STRING), 3, '0'))  AS session_id,
       MIN(event_ts) AS session_start,
       MAX(event_ts) AS session_end,
       COUNT(*)      AS event_count,
       ROUND((unix_timestamp(MAX(event_ts)) - unix_timestamp(MIN(event_ts))) / 60.0, 1) AS duration_min
FROM sessions
GROUP BY user_id, session_seq
ORDER BY user_id, session_start
""")
result.show(truncate=False)

# Expected: u1 has 2 sessions (3 events + 2 events), u2 has 2 sessions.
# Talking points for the interviewer:
# 1) The gap flag (is_new_session) marks the START of each island; the running SUM turns
#    flags into a dense island id (session_seq) — no self-join needed, O(n) over the partition.
# 2) Alternative classic trick: island_id = row_number() OVER (overall order)
#       - row_number() OVER (partition by user order by ts) — subtract the two; equal values
#    within an island form the island key. Same cost, less readable.
# 3) Scale note: on billions of events, partition the source by user_id (or bucket by it) so
#    the window never shuffles the same user's events across executors twice.

## Q2 [ADVANCED] — Cohort Analysis: Monthly Signup Retention

Given a `users` table (user_id, signup_date) and an `orders` table (order_id, user_id, order_date), build a **monthly signup-cohort retention matrix**:

1. Assign every user to a cohort = `date_trunc('month', signup_date)`.
2. For each user, compute the number of complete months between signup and their first order (`months_between`).
3. Output one row per (cohort_month, months_since_signup) with: active users in that month, total users in the cohort, and retention rate (rounded to 2 decimals). Only include months 0-3.

**Interview points:** Cohort analysis is a top-5 product-analytics SQL question (Meta, Uber, Airbnb all ask variants). The two hard parts are joining each user to *every* month of their activity and picking the right window/join strategy. Explain why a plain `users JOIN orders` under-counts users with no order (retention month 0 must still show them).

In [0]:
# Solution Q2 — Monthly signup-cohort retention matrix
# Strategy: LEFT JOIN users -> first order (users with no order must survive for month-0 retention).
from pyspark.sql import functions as F, Window

users = [
    (1, "2026-01-05"), (2, "2026-01-20"), (3, "2026-01-28"),  # Jan cohort
    (4, "2026-02-03"), (5, "2026-02-15"), (6, "2026-02-27"),  # Feb cohort
    (7, "2026-03-10"),                                        # Mar cohort
]
orders = [
    (101, 1, "2026-01-06"), (102, 1, "2026-02-10"), (103, 1, "2026-04-15"),
    (104, 2, "2026-01-21"),
    (105, 3, "2026-02-01"), (106, 3, "2026-03-05"), (107, 3, "2026-04-20"),
    (108, 4, "2026-02-04"), (109, 4, "2026-04-01"),
    (110, 5, "2026-02-16"),
    (111, 7, "2026-03-11"), (112, 7, "2026-04-12"),
]
users_df = spark.createDataFrame(users, ["user_id", "signup_date"]).withColumn("signup_date", F.to_date("signup_date"))
orders_df = spark.createDataFrame(orders, ["order_id", "user_id", "order_date"]).withColumn("order_date", F.to_date("order_date"))
users_df.createOrReplaceTempView("users")
orders_df.createOrReplaceTempView("orders")

# NOTE: correlated scalar subqueries are not supported in join conditions on this runtime,
# so we compute the first order per user with row_number() — which is also the scalable way.
cohort = spark.sql("""
WITH joined AS (
  SELECT u.user_id,
         u.signup_date,
         date_trunc('month', u.signup_date) AS cohort_month,
         o.order_date
  FROM users u
  LEFT JOIN orders o ON o.user_id = u.user_id
),
tagged AS (
  SELECT *,
         row_number() OVER (PARTITION BY user_id ORDER BY order_date) AS rn,
         COUNT(order_date) OVER (PARTITION BY user_id)          AS n_orders
  FROM joined
),
first_order AS (
  SELECT user_id, cohort_month,
         floor(months_between(order_date, signup_date)) AS months_since_signup
  FROM tagged
  WHERE rn = 1 AND order_date IS NOT NULL
),
never_ordered AS (
  SELECT user_id, cohort_month, -1 AS months_since_signup
  FROM tagged WHERE n_orders = 0
),
per_user AS (
  SELECT * FROM first_order
  UNION ALL
  SELECT * FROM never_ordered
),
cohort_sizes AS (
  SELECT date_trunc('month', signup_date) AS cohort_month, COUNT(*) AS cohort_size
  FROM users GROUP BY 1
)
SELECT p.cohort_month,
       CAST(p.months_since_signup AS INT) AS months_since_signup,
       COUNT(*)                     AS active_users,
       MAX(c.cohort_size)           AS cohort_size,
       ROUND(COUNT(*) / MAX(c.cohort_size), 2) AS retention_rate
FROM per_user p
JOIN cohort_sizes c ON c.cohort_month = p.cohort_month
GROUP BY p.cohort_month, months_since_signup
ORDER BY p.cohort_month, months_since_signup
""")
cohort.show(truncate=False)

# Notes to say out loud:
# - months_since_signup = -1 rows are users who NEVER ordered (they show up because of the
#   LEFT JOIN) — in an interview ask whether "retention month 0" means 'signed up' or 'ordered
#   in month 0'; here we keep both views available.
# - The correlated subquery for the first order is fine on small data; on real scale you would
#   pre-compute first_order per user with row_number() OVER (PARTITION BY user_id ORDER BY order_date)
#   = 1 in a CTE and join on that (one shuffle instead of a per-row subquery).
# - Classic pivot for the final matrix: cohort_month as rows, months_since_signup as columns,
#   retention_rate as value — done with CASE WHEN per month or pivot() in PySpark.

In [0]:
## Q3 [INTERMEDIATE] — Design an LRU Cache

Implement an LRU (Least Recently Used) cache with O(1) `get` and `put`:

* `LRUCache(capacity)` — construct the cache.
* `get(key)` → return the value or -1 if missing. A `get` makes the key the most recently used.
* `put(key, value)` → insert/update. If capacity is exceeded, **evict the least recently used key**.

Follow-up (what the interviewer will ask): Can you do it without `collections.OrderedDict`? (Use a hash map + doubly linked list.) Why is a plain dict + list too slow? (list eviction/lookup is O(n).)

**Interview points:** This is a real LeetCode-medium (LC 146) asked at Amazon, Uber, and Flipkart constantly. Talk through the invariant: *the data structure's ORDER must encode recency, and both operations must touch order in O(1).*

In [0]:
# Solution Q3 — LRU cache with OrderedDict + a from-scratch doubly-linked-list version
from collections import OrderedDict

class LRUCache:
    """O(1) get/put using OrderedDict (move_to_end reorders in O(1))."""
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)              # mark most recently used
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)        # pop LRU (oldest) end

c = LRUCache(2)
c.put(1, 1); c.put(2, 2)
assert c.get(1) == 1        # 1 is now MRU
c.put(3, 3)                  # evicts key 2 (LRU)
assert c.get(2) == -1
c.put(4, 4)                  # evicts key 1
assert c.get(1) == -1
assert c.get(3) == 3 and c.get(4) == 4

# --- Follow-up: hash map + doubly linked list (no OrderedDict) ---
class _Node:
    __slots__ = ("key", "value", "prev", "next")
    def __init__(self, key=None, value=None):
        self.key, self.value = key, value
        self.prev = self.next = None

class LRUCacheFromScratch:
    """head <-> MRU ... LRU <-> tail. Map: key -> node. All ops O(1)."""
    def __init__(self, capacity):
        self.cap = capacity
        self.map = {}
        self.head, self.tail = _Node(), _Node()   # sentinels
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node):
        node.prev.next, node.next.prev = node.next, node.prev

    def _add_front(self, node):                   # insert right after head = MRU
        node.next, node.prev = self.head.next, self.head
        self.head.next.prev = node
        self.head.next = node

    def get(self, key):
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node); self._add_front(node)  # refresh recency
        return node.value

    def put(self, key, value):
        if key in self.map:
            self._remove(self.map[key])
        node = _Node(key, value)
        self.map[key] = node
        self._add_front(node)
        if len(self.map) > self.cap:                # evict node before tail (LRU)
            lru = self.tail.prev
            self._remove(lru)
            del self.map[lru.key]

c2 = LRUCacheFromScratch(2)
c2.put(1, "a"); c2.put(2, "b")
assert c2.get(1) == "a"
c2.put(3, "c")
assert c2.get(2) == -1 and c2.get(1) == "a" and c2.get(3) == "c"
print("Q3 all asserts passed — LRU cache works in O(1) per op.")

# Complexity: get/put are O(1) time, O(capacity) space.
# Real-world tie-in: functools.lru_cache does exactly this for Python functions;
# Spark's block manager cache eviction and CPU caches are all LRU variants.

## Q4 [ADVANCED] — Graph Traversal: Shortest Path with BFS

You are given a social network as an edge list: for each pair `(a, b)` there is a bidirectional friendship. Given a `start` user and an `end` user, return **the shortest friendship path** (list of users) between them, and its length (degrees of separation). If unreachable, return an empty path.

1. Implement it with BFS and explain why BFS (not DFS) gives the shortest path in an unweighted graph.
2. Follow-up: how would you find the distance from one user to *all* users (single-source shortest path), and what changes when edges have weights (Dijkstra)?

**Interview points:** Graph traversal with BFS is a genuine FAANG staple (Meta loves "degrees of separation"). Key things to nail: building an adjacency list from an edge list, tracking visited nodes to avoid cycles, and storing the *path* via a parent map rather than the queue.

In [0]:
# Solution Q4 — BFS shortest path on an undirected graph
from collections import deque, defaultdict

def shortest_path(edges, start, end):
    # 1) Build adjacency list (undirected -> add both directions)
    graph = defaultdict(list)
    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)

    # 2) BFS from start; parent map lets us rebuild the path afterwards
    parent = {start: None}
    queue = deque([start])
    while queue:
        node = queue.popleft()
        if node == end:                 # early exit the moment we reach end
            break
        for nbr in graph[node]:
            if nbr not in parent:       # unvisited -> record parent, enqueue
                parent[nbr] = node
                queue.append(nbr)

    if end not in parent:
        return []                       # unreachable

    # 3) Walk the parent chain backwards end -> start, then reverse
    path, node = [], end
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

edges = [("A","B"), ("A","C"), ("B","D"), ("C","D"), ("D","E"), ("C","F"), ("F","E")]
path_ae = shortest_path(edges, "A", "E")
assert path_ae[0] == "A" and path_ae[-1] == "E" and len(path_ae) == 4  # any 3-hop path is a valid shortest
assert len(shortest_path(edges, "A", "E")) == 4          # 3 hops
assert shortest_path(edges, "A", "A") == ["A"]
assert shortest_path(edges + [("X","Y")], "A", "X") == []  # disconnected
assert shortest_path(edges, "A", "Z") == []               # unknown node
print("Q4 asserts passed. Example path A->E:", shortest_path(edges, "A", "E"))

# Why BFS gives shortest paths: it explores nodes in order of increasing distance from the
# source (level by level). The FIRST time we reach a node, we reached it via a minimum number
# of edges — later arrivals can never be shorter. DFS dives deep, so it may find a longer path
# first and has no level guarantee.
#
# Follow-ups to state out loud:
# - All-users distances: run the same BFS but never early-exit; distance for every reachable
#   node falls out in O(V + E).
# - Weighted edges: BFS's level guarantee breaks -> use Dijkstra (priority queue, O(E log V));
#   negative weights require Bellman-Ford.
# - At social-network scale you do not run BFS per pair: bidirectional BFS (expand from both
#   ends, meet in the middle) or precomputed landmark distances cut the search space massively.
# Complexity: O(V + E) time, O(V) space.

## Q5 [INTERMEDIATE] — CDC Processing: Apply a Change Log to a Target Table

You are building a **CDC (Change Data Capture) pipeline**. The source publishes an ordered change log with columns: `(order_id, status, updated_at, op, _commit_ts)` where `op ∈ {'I' (insert), 'U' (update), 'D' (delete)}` and `_commit_ts` is the source commit timestamp.

Process the log into a `target_orders` table so that:

1. Only the **latest change per order_id** (by `_commit_ts`) is applied — out-of-order and duplicate events must not corrupt state.
2. `D` events remove the row, `I`/`U` upsert the latest values.
3. The solution must be **re-runnable** (idempotent) — replaying the same log changes nothing.

**Interview points:** The universal CDC pattern is *dedup-first, then merge*: take latest-per-key with `row_number()`, then `MERGE INTO` with delete handling. Say explicitly why you cannot just apply events in arrival order (late/duplicated events).

In [0]:
# Solution Q5 — CDC: dedup to latest-per-key, then MERGE with delete handling
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
    StructField("order_id", StringType()), StructField("status", StringType()),
    StructField("updated_at", StringType()), StructField("op", StringType()),
    StructField("_commit_ts", StringType()),
])
cdc_events = [
    # order o1: insert -> update -> LATE duplicate of the insert (must NOT win)
    ("o1", "CREATED",  "2026-01-01 10:00", "I", "2026-01-01 10:00:05"),
    ("o1", "SHIPPED",  "2026-01-02 09:00", "U", "2026-01-02 09:00:05"),
    ("o1", "CREATED",  "2026-01-01 10:00", "I", "2026-01-01 10:00:05"),  # dup, earlier commit
    # order o2: insert then delete
    ("o2", "CREATED",  "2026-01-01 11:00", "I", "2026-01-01 11:00:05"),
    ("o2", None,       None,                "D", "2026-01-03 08:00:05"),
    # order o3: insert only
    ("o3", "CREATED",  "2026-01-02 12:00", "I", "2026-01-02 12:00:05"),
]
raw = (spark.createDataFrame(cdc_events, schema)
       .withColumn("updated_at", F.to_timestamp("updated_at"))
       .withColumn("_commit_ts", F.to_timestamp("_commit_ts")))

# Step 1: latest event per key (ties broken by op so a delete beats an update on equal ts)
latest = (raw
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("order_id").orderBy(F.col("_commit_ts").desc(), F.col("op").desc())))
    .filter("rn = 1").drop("rn"))
print("Latest event per order:")
latest.select("order_id", "status", "op", "_commit_ts").show(truncate=False)

# Step 2: split upserts vs deletes
upserts = latest.filter("op IN ('I','U')").select("order_id", "status", "updated_at")
deletes  = latest.filter("op = 'D'").select("order_id")

# Step 3: idempotent apply with MERGE (drop + recreate the demo target so re-runs are safe)
spark.sql("DROP TABLE IF EXISTS day08_demo.target_orders")
upserts.write.format("delta").saveAsTable("day08_demo.target_orders")
upserts.createOrReplaceTempView("src_upserts")
spark.sql("""
MERGE INTO day08_demo.target_orders t
USING src_upserts s ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET t.status = s.status, t.updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (order_id, status, updated_at) VALUES (s.order_id, s.status, s.updated_at)
""")
deletes.createOrReplaceTempView("src_deletes")
spark.sql("""
MERGE INTO day08_demo.target_orders t
USING src_deletes s ON t.order_id = s.order_id
WHEN MATCHED THEN DELETE
""")

print("Target table after CDC apply:")
spark.sql("SELECT * FROM day08_demo.target_orders ORDER BY order_id").show(truncate=False)

# Expected: o1 SHIPPED (latest commit wins, not the duplicate CREATED), o2 gone (deleted), o3 CREATED.
# Re-running this whole cell is idempotent: DROP+recreate + MERGE on identical inputs -> identical state.
# Interview follow-up: in production the dedup window reads only the INCREMENT since the last
# processed _commit_ts (stored in a checkpoint table), and Delta's MERGE gives atomic apply —
# partial batches never leave the target half-updated.

In [0]:
## Q6 [ADVANCED] — Skewed Join in Code: Salting a Hot Key

A `orders` table has a badly skewed key: 60% of rows belong to `store_id = 100` (one giant store). Joining `orders` to `stores` on `store_id` in a sort-merge join causes one straggler task to run for ages while every other task finishes in seconds.

Fix it **in code** with the **salting technique**: replicate the small dimension rows N times with a salt suffix, explode the fact rows' keys with a random salt, and join on `(key, salt)`. Then demonstrate the before/after partition balance.

**Interview points:** Be ready to explain WHY salting works: it converts 1 hot hash bucket into N evenly-filled buckets by forcing hot-key rows to spread across `spark.sql.shuffle.partitions` buckets. Also mention the alternatives — AQE skew-join handling (automatic), broadcast join (if the dim fits in memory), and isolate hot keys into a separate broadcast branch.

In [0]:
# Solution Q6 — salted join for a hot key
from pyspark.sql import functions as F, Window
import random

N_SALTS = 8  # number of salt buckets for the hot key

orders_rows = [("o%03d" % i, 100 if i % 5 < 3 else i % 5, i * 10.0) for i in range(50)]  # ~60% store 100
stores_rows = [(100, "MegaMart")] + [(s, f"Store_{s}") for s in range(5)]

orders = spark.createDataFrame(orders_rows, ["order_id", "store_id", "amount"])
stores = spark.createDataFrame(stores_rows, ["store_id", "store_name"])

print("Rows per store_id BEFORE salting (the skew):")
orders.groupBy("store_id").count().orderBy(F.desc("count")).show(3)

# --- Salting ---
# 1) Fact side: append a random salt in [0, N) to the (already skewed) join key
salted_orders = (orders
    .withColumn("salt", (F.rand() * N_SALTS).cast("int"))
    .withColumn("salted_store_id", F.concat_ws("#", F.col("store_id").cast("string"), F.col("salt").cast("string"))))

# 2) Dim side: replicate each store N times, one per salt value
salted_stores = (stores
    .withColumn("salt", F.explode(F.sequence(F.lit(0), F.lit(N_SALTS - 1))))
    .withColumn("salted_store_id", F.concat_ws("#", F.col("store_id").cast("string"), F.col("salt").cast("string"))))

# 3) Join on the salted key — hot key 100 now spreads across N buckets
joined = (salted_orders.alias("o")
    .join(salted_stores.alias("s").drop("salt"), "salted_store_id")
    .select("o.order_id", "o.store_id", "s.store_name", "o.amount"))

print("Salted join result (correctness check — every order matched):")
joined.orderBy("order_id").show(5, truncate=False)
n_joined = joined.count()
assert n_joined == orders.count(), f"salting lost rows: {n_joined} != {orders.count()}"
print(f"All {n_joined} orders joined — no row loss, duplicates impossible (fact has exactly one salt).")

# Why it fixes the straggler:
# BEFORE: hash(store_id=100) lands in ONE shuffle bucket -> that task processes 30 rows while
#         others process ~4. At scale that one task hours while others finish in minutes.
# AFTER:  store 100's rows split across 8 buckets (~4 rows each at this scale; even split at scale),
#         so no single bucket dominates. Dim replication cost: |dim| x N_SALTS rows (tiny).
# Alternatives to mention:
#   1) spark.sql.adaptive.skewJoin.enabled=true  — AQE splits skewed partitions automatically (no code).
#   2) broadcast(stores)                          — if the dim fits in driver+executor memory, no shuffle at all.
#   3) Isolate hot keys: join hot-key rows via broadcast, the rest via SM join (two branches, union).

## Q7 [ADVANCED] — AQE Internals: What Really Happens Under the Hood

"You said AQE fixed your job — walk me through exactly what Adaptive Query Execution does, stage by stage, and which three optimizations it applies. When does AQE help and when is it a no-op?"

Cover in your answer:

1. How AQE changes the execution model from static one-shot planning to **re-planning at shuffle boundaries** (what a `QueryStage` is).
2. The three headline optimizations: **partition coalescing**, **skew-join handling**, and **join strategy switch (SMJ → broadcast)**.
3. Which configs control each, and what the Spark UI shows you when AQE fired.

**Interview points:** AQE internals is a senior-level Spark question that separates people who used a config from people who understand the engine. The key insight to state: *AQE works because shuffle map output statistics (rows/size per partition) are exact, not estimates — so planning decisions made after a shuffle are based on truth.*

In [0]:
# Solution Q7 — AQE internals explained (explanatory)

print("""
AQE (Adaptive Query Execution) — Spark 3.0+, on by default since 3.2 — re-optimizes a query
DURING execution at every shuffle boundary instead of trusting up-front estimates.

THE EXECUTION MODEL CHANGE
--------------------------
Classic Spark: plan the whole query once with cardinality ESTIMATES (from table stats/histograms),
then execute blindly. One bad estimate early poisons every downstream decision.
AQE:           the query is decomposed into QueryStages. A QueryStage is a sub-DAG that ends in a
               shuffle (materialized exchange). Stages run in dependency order; when a stage
               finishes, its ACTUAL per-partition statistics (row count, data size) are known.
               The planner then re-optimizes all stages that consume it — with true numbers.

THE THREE HEADLINE OPTIMIZATIONS
--------------------------------
1) Dynamic partition coalescing (shuffle partition merging)
   Config: spark.sql.adaptive.coalescePartitions.enabled (default true), with
           advisoryPartitionSizeInBytes (default 64MB) as the target size.
   Problem solved: you set shuffle.partitions=200, but the shuffle only produced 50MB total ->
   200 tiny tasks, each ~0.25MB, scheduling overhead dominates.
   What AQE does: AFTER the shuffle, it sees actual partition sizes and coalesces them into
   fewer, ~64MB partitions. 200 tiny tasks become e.g. 2 well-sized tasks. Initial
   partition num stays an upper bound, never a commitment.

2) Dynamically switching join strategies (SMJ -> BroadcastHashJoin)
   Config: spark.sql.adaptive.autoBroadcastJoinThreshold (default same as static 10MB).
   Problem solved: stats estimated the right side at 500MB -> SortMergeJoin; after running the
   right side it turns out to be 5MB -> broadcast would have been far cheaper (no shuffle,
   no sort). AQE flips the join to broadcast using the MEASURED size.

3) Skewed join handling (splitting skewed partitions)
   Config: spark.sql.adaptive.skewJoin.enabled (default true) + skewedPartitionFactor (5x)
           + skewedPartitionThresholdInBytes (256MB).
   Problem solved: one join key hashes to one giant partition -> straggler task.
   What AQE does: partitions exceeding median-size * factor AND the 256MB threshold get SPLIT
   into roughly equal sub-partitions (partition A becomes A_0, A_1, ...); the matching side
   is replicated to each sub-partition so the join result stays correct. The straggler
   becomes several normal tasks running in parallel.

WHEN AQE IS A NO-OP / CAN'T HELP
--------------------------------
- No shuffle in the plan (single-stage map-only job): nothing to re-plan on.
- Filters were selective BEFORE the shuffle anyway: coalescing is all you get.
- Skew on an AGGREGATION key (not a join key): AQE skew handling targets joins; skewed
  groupBy still needs salting in code.
- Table stats wildly wrong + broadcast: still risky — the measured size arrives after the
  build side runs, so a join right after a scan with no intermediate shuffle decides early.

WHAT THE SPARK UI SHOWS YOU
---------------------------
- The query plan shows QueryStage boundaries and plans annotated with 'AdaptiveSparkPlan'
  + 'isFinalPlan=true'. You can see the initial plan and the final plan side by side.
- Skew fix visible as the partition count of one reducer being replaced by multiple
  sub-partitions; coalescing visible as fewer post-shuffle tasks than shuffle.partitions.

CONFIG SNAPSHOT (note: serverless/managed Spark may manage these for you)
-------------------------------------------------------------------------
spark.sql.adaptive.enabled = true
spark.sql.adaptive.coalescePartitions.initialPartitionNum = 200   # upper bound
spark.sql.adaptive.advisoryPartitionSizeInBytes = 64MB
spark.sql.adaptive.autoBroadcastJoinThreshold = 10MB
spark.sql.adaptive.skewJoin.enabled = true
spark.sql.adaptive.skewJoin.skewedPartitionFactor = 5
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 256MB
""")

# One-line summary to end the interview answer:
print("AQE = 'plan with the truth you learn along the way': every shuffle boundary is a free re-planning "
      "opportunity using exact stats — so it fixes over-partitioning, wrong join strategy, and join skew "
      "without you touching the code.")

## Q8 [ADVANCED] — Speculative Execution: Fixing Stragglers Without Touching the Data

"One task in your 500-task stage has been running for 40 minutes; the other 499 finished in 3 minutes. The task is NOT failing — it is just slow (bad disk, noisy executor). Explain **speculative execution**: how it works, when to enable it, its risks with non-idempotent output, and how it interacts with Delta Lake writes."

Cover: the straggler problem, `spark.speculation.*` configs, the 75%-heuristic trigger, why tasks must be idempotent, and why speculative execution is disabled or risky for tasks with side effects (external writes, non-committed sinks).

**Interview points:** This question tests whether you understand *task-level* fault tolerance vs *job-level* retries — a distinction senior engineers are expected to draw instantly.

In [0]:
# Solution Q8 — Speculative execution (explanatory, with config demo)
import time

print("""
SPECULATIVE EXECUTION — how it actually works
----------------------------------------------
Problem: in a stage of N tasks, one straggler (failing disk, noisy neighbor, GC pressure,
unlucky data locality) can hold the whole stage hostage. You cannot restart it — it IS
succeeding, just slowly.
Mechanism: when enabled, the driver monitors running tasks. If a task is still running while
> speculativeQuantile (default 0.75) of the stage's tasks have ALREADY FINISHED, AND the
running task is slower than the median completion time by more than speculativeMultiplier
(default 1.5x), the driver launches a SPECULATIVE COPY of that task on a DIFFERENT executor.
Whichever instance finishes FIRST wins; the loser is killed and its partial work discarded.

The key configs
---------------
spark.speculation                      = true      # master switch (default false)
spark.speculation.interval              = 100ms     # how often the driver checks
spark.speculation.quantile              = 0.75      # >=75% of stage's tasks done before considering
spark.speculation.multiplier            = 1.5       # task must be 1.5x slower than median
spark.speculation.minTaskRuntime...                 # ignore tasks that finish trivially fast

What it costs you
-----------------
- Extra cluster resources: you run the speculative copy IN ADDITION to the original until one wins.
- Task idempotence is mandatory: two copies may write side effects twice. With file/idempotent
  sinks (same task id -> same output location, write + commit marker) this is safe BECAUSE only
  the winner's commit is accepted. With foreachBatch / external REST calls / JDBC upserts it is
  NOT safe unless you make the write idempotent (upsert by key, dedupe on arrival).
- On structured streaming sinks, speculation is typically disabled by the sink contract
  (exactly-once sinks rely on a single writer per task).

When to enable
---------------
- Heterogeneous clusters / shared infra / long map-heavy stages where stragglers dominate runtime.
- Delta Lake batch MERGE/write jobs: safe (transactional commit deduplicates the loser's files).
- NOT for: streaming foreachBatch with non-idempotent external effects, side-effectful UDFs
  (send email, charge card) unless made idempotent.

Related, and the distinction interviewers want:
- Speculative execution = TASK-level, proactive, for SLOW (not failed) tasks.
- Task retries (spark.task.maxFailures, default 4) = TASK-level, reactive, for FAILED tasks.
- Job/stage retries (Jobs UI 'Run if FAILED', Repair Runs re-running a failed job from the
  failed task onwards) = JOB-level, reactive.
A senior answer names all three layers and when each one fires.
""")

# Tiny demonstration of the trigger heuristic (pure Python mirror of the driver logic):
def would_speculate(completion_times_ms, still_running_ms):
    done = [t for t in completion_times_ms]
    if len(done) == 0:
        return False
    done.sort()
    median = done[len(done) // 2]
    slow = still_running_ms > 1.5 * median  # multiplier
    return slow

tasks_done_ms = [3000, 3100, 2900, 3300, 3050]  # 499 tasks ~3s
assert would_speculate(tasks_done_ms, 40 * 60 * 1000) is True   # 40-min straggler
assert would_speculate(tasks_done_ms, 3500) is False            # normal task
print("Speculation heuristic demo: 40-min straggler on a 3s-median stage -> speculative copy launched.")

In [0]:
## Q9 [ADVANCED] — Exactly-Once Semantics: End-to-End Delivery Guarantees

"Your streaming pipeline from Kafka to a Delta table has duplicated rows after a restart, and your PM is asking if it lost rows too. Explain the delivery semantics spectrum (at-most-once, at-least-once, exactly-once), **where duplicates actually come from**, and design an exactly-once pipeline: Kafka source → Structured Streaming → Delta sink. Show the code that makes replay safe."

**Interview points:** The trap answer is "set a config and it's exactly-once." The strong answer: *exactly-once is not a single feature — it's at-least-once delivery + an idempotent/transactional sink.* Name the failure window (offset committed after write = duplicates; before write = loss) and how checkpoint + Delta transactional MERGE close it.

In [0]:
# Solution Q9 — exactly-once = at-least-once delivery + idempotent sink (dedup + MERGE)
from pyspark.sql import functions as F, Window

print("""
THE SEMANTICS SPECTRUM (and where each breaks)
-----------------------------------------------
At-most-once:  commit offset FIRST, then process. Crash between -> those records are LOST.
At-least-once: process FIRST, commit offset LAST. Crash between -> records are REPLAYED (duplicates).
               This is what Structured Streaming gives by default (checkpointed offsets).
Exactly-once:  at-least-once delivery + a sink that makes REPLAY A NO-OP.
               Two ways to get the no-op:
                 a) Transactional sink: writes + offset commit happen in ONE atomic transaction.
                    The Delta streaming sink does this internally — each streaming commit's
                    transaction-log entry records the batch id; on restart the engine resumes
                    from the checkpoint and skips already-committed batches.
                 b) Idempotent sink keyed by business key: dedup within the batch by source
                    identity (topic, partition, offset) or event id, then MERGE (upsert) by
                    key — a replay either finds nothing new or produces identical row state.

WHERE DUPLICATES ACTUALLY COME FROM
------------------------------------
Not usually the engine — usually: reprocessing from earliest after a checkpoint loss, a
parallel batch backfill overlapping the stream, two stream instances sharing (or NOT sharing)
a checkpoint location, or a source that redelivers (at-least-once producers).
""")

# --- The code that makes replay safe: per-(partition, offset) dedup + MERGE by key ---
kafka_like_batch = [  # one micro-batch where the source redelivered two records
    ("k1", "order_created", 100.0, "p0", 11),
    ("k2", "order_created",  50.0, "p0", 12),
    ("k1", "order_created", 100.0, "p0", 11),  # redelivery of offset 11
    ("k3", "order_paid",     100.0, "p0", 13),
    ("k2", "order_created",  50.0, "p0", 12),  # redelivery of offset 12
]
bdf = spark.createDataFrame(kafka_like_batch,
        ["event_key", "event_type", "amount", "src_partition", "src_offset"])

# 1) Deterministic dedup within the batch on the source identity (partition, offset)
deduped = (bdf
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("src_partition", "src_offset").orderBy("src_offset")))
    .filter("rn = 1").drop("rn"))
assert deduped.count() == 3, "dedup must collapse redelivered offsets"
print(f"\nBatch of {bdf.count()} raw records deduped to {deduped.count()} unique events.")

# 2) Idempotent MERGE into Delta keyed by event identity — replay = no-op
spark.sql("DROP TABLE IF EXISTS day08_demo.events_exactly_once")
deduped.select("event_key", "event_type", "amount").write.format("delta").saveAsTable("day08_demo.events_exactly_once")
deduped.createOrReplaceTempView("src_events")
merge_sql = """
MERGE INTO day08_demo.events_exactly_once t
USING src_events s ON t.event_key = s.event_key
WHEN MATCHED THEN UPDATE SET t.event_type = s.event_type, t.amount = s.amount
WHEN NOT MATCHED THEN INSERT (event_key, event_type, amount) VALUES (s.event_key, s.event_type, s.amount)
"""
spark.sql(merge_sql)

# 3) Simulate the replay: the same batch arrives again after a crash
spark.sql(merge_sql)
n_after_replay = spark.sql("SELECT COUNT(*) AS c FROM day08_demo.events_exactly_once").first()[0]
assert n_after_replay == 3, "replay must not duplicate rows"
print(f"Replayed the same batch: table still has {n_after_replay} rows — exactly-once achieved.")

# Interview closing line: the streaming checkpoint gives at-least-once; the Delta sink's
# transactional commit (streaming batch id in the txn log) or your dedup+MERGE turns it
# into exactly-once. Never claim a single config did it.

## Q10 [ADVANCED] — Late Data & Watermarks: Grace Windows in Streaming

You stream taxi ride events (ride_id, city, event_time). Events can arrive up to **15 minutes late** (phone offline, network retries). Requirements:

1. Maintain a **per-city 10-minute tumbling window count** of rides.
2. Accept events that are correctly-late, but never hold state forever: aggregate windows older than the watermark must be finalized and evicted.
3. State must not grow unboundedly.

Explain the **event-time watermark**, how `withWatermark(...)` + window aggregation interact, and what `withWatermark('event_time', '15 minutes')` actually guarantees. Show the code, then explain for a specific example which late event survives and which is dropped.

**Interview points:** The subtlety everyone gets wrong: the watermark is computed as `max(event_time) - threshold` **seen so far in data order**, not wall clock. Also be precise that *late-dropped rows never appear in the output at all* — they vanish silently, which is itself a monitoring concern.

In [0]:
# Solution Q10 — watermark + late-data grace windows in Structured Streaming
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
import datetime as dt

print("""
HOW WATERMARKS WORK
--------------------
withWatermark('event_time', '15 minutes') tells the engine: 'events up to 15 min older than
the MAX event_time observed so far may still be relevant — keep window state for them;
anything older than max(event_time) - 15 min is LATE-DROPPED forever.'

What the watermark drives:
1. State eviction: once the watermark passes a window's end, that window can never receive
   new rows -> its aggregate is finalized and its state dropped -> bounded memory.
2. Late-drop: rows arriving with event_time below the watermark are DISCARDED SILENTLY.

Common misconception: the watermark tracks wall-clock time. It does NOT — it derives from
max(event_time) seen in the DATA. If the stream stalls, the watermark stalls too.
""")

# Demo: windowed aggregation with a watermark (batch mirror of the streaming logic)
schema = StructType([
    StructField("ride_id", StringType()), StructField("city", StringType()),
    StructField("event_time", TimestampType()),
])
base = dt.datetime(2026, 1, 1, 10, 0, 0)
events = [
    ("r1", "Delhi",  base),                        # window 10:00-10:10 Delhi
    ("r2", "Delhi",  base + dt.timedelta(minutes=9)),
    ("r3", "Mumbai", base + dt.timedelta(minutes=20)),  # max event_time -> 10:20
    ("r4", "Delhi",  base + dt.timedelta(minutes=21)),  # max event_time -> 10:21
]
df = spark.createDataFrame(events, schema)

agg = (df
    .withWatermark("event_time", "15 minutes")   # in streaming this bounds state
    .groupBy(F.window("event_time", "10 minutes").alias("w"), "city")
      .agg(F.count("*").alias("rides"))
    .select(F.col("w.start").alias("window_start"), F.col("w.end").alias("window_end"),
            "city", "rides")
    .orderBy("window_start", "city"))
agg.show(truncate=False)

# Late-drop illustration: after seeing event_time 10:21, watermark = 10:21 - 15 min = 10:06.
# A new Delhi event with event_time 10:02 is 19 min late -> 10:02 < 10:06 -> DROPPED.
late_event_time = base + dt.timedelta(minutes=2)
max_seen = base + dt.timedelta(minutes=21)
watermark = max_seen - dt.timedelta(minutes=15)
survives = late_event_time >= watermark
print(f"Late event at {late_event_time.time()} vs watermark {watermark.time()} -> "
      f"{'ACCEPTED' if survives else 'DROPPED (silently!)'}")
assert survives is False

# Points to raise with the interviewer:
# - Grace period == watermark threshold: '15 min late' tolerance == withWatermark 15 minutes.
# - Output mode: append emits a window only AFTER the watermark passes its end (bounded
#   latency); update mode emits running counts every trigger.
# - Stream-stream joins use the MIN of both sides' watermarks — the slower side pins the
#   join's state lifetime (classic memory blow-up follow-up).
# - Monitoring: track watermark lag vs wall clock and dropped-late-records metrics — a stalled
#   source stalls the watermark and delays every downstream window.

## Q11 [INTERMEDIATE] — Delta Lake Layout: OPTIMIZE, ZORDER and the Small-File Problem

"Your Delta table has 200,000 tiny files because a streaming job commits every 10 seconds. Reads are slow and `MERGE` performance degrades weekly. Explain:

1. Why many small files hurt (metadata pressure, open-file cost, file listing, scan efficiency).
2. What `OPTIMIZE` does (bin-packing compaction) vs what ZORDER does (multi-dimensional clustering) — and why they are different tools.
3. How you would schedule maintenance on a production table, and what `VACUUM` has to do with it."

Then write the code to demonstrate OPTIMIZE on a small Delta table and show the file count dropping.

**Interview points:** The senior details: OPTIMIZE is safe to run concurrently with readers/writers (Delta snapshot isolation — readers keep reading the old snapshot), and ZORDER only pays off if queries actually filter on the ZORDER columns — it's not free (full rewrite).

In [0]:
# Solution Q11 — small-file compaction: OPTIMIZE + ZORDER + VACUUM
from pyspark.sql import functions as F

print("""
WHY SMALL FILES HURT
--------------------
1. Transaction log: every commit appends file entries to the Delta log; planning a query over
   200k files is slow BEFORE any data is read.
2. Open-file overhead: each file costs an open + footer read on S3/ADLS (tens of ms) — 200k
   files can burn minutes on overhead alone.
3. Scan inefficiency: tiny per-task work cannot amortize scheduling cost.
4. MERGE: more files = more candidate files to prune/rewrite per commit -> compounding decay.

OPTIMIZE (bin-packing)   : rewrites small files into ~targetFileSize (1GB default) files per
   partition. Layout-only, data unchanged, snapshot-isolated (concurrent-safe).
ZORDER (multi-dim cluster): interleaves row order so values of ALL zordered columns spread
   evenly across files -> data skipping works for predicates on ANY of them. Use when queries
   filter on several non-partition columns (customer_id AND event_date). Cost: full rewrite.
VACUUM: deletes files unreferenced by any snapshot older than retention (default 7 days).
   OPTIMIZE leaves the old small files referenced by time travel until expiry -> VACUUM
   reclaims the space. Order matters: OPTIMIZE first, VACUUM after retention passes.
""")

# --- Demo: create small files, then compact them ---
spark.sql("DROP TABLE IF EXISTS day08_demo.small_files")
(spark.range(20000)
     .withColumn("grp", (F.col("id") % 20).cast("string"))
     .withColumn("payload", F.lpad(F.col("id").cast("string"), 400, "x"))
     .write.format("delta").partitionBy("grp").saveAsTable("day08_demo.small_files"))

# Simulate streaming micro-commits: 10 small appends
for i in range(10):
    spark.sql("INSERT INTO day08_demo.small_files SELECT id, grp, payload FROM day08_demo.small_files WHERE id < 200")

def num_files():
    return spark.sql("DESCRIBE DETAIL day08_demo.small_files").select("numFiles").first()[0]

before = num_files()
spark.sql("OPTIMIZE day08_demo.small_files").collect()   # bin-packing compaction
after = num_files()
print(f"\nnumFiles: after micro-appends = {before}  ->  after OPTIMIZE = {after}")
assert after <= before

# Production maintenance (say this out loud):
# - Daily off-peak OPTIMIZE on hot tables (or enable predictive optimization on UC tables).
# - ZORDER only on columns real queries filter on; re-run after large backfills.
# - VACUUM weekly with explicit retention no shorter than your restore/time-travel SLA.
# - Also reduce small-file creation at the SOURCE (longer trigger intervals, availableNow
#   triggers) — compaction treats the symptom, not the cause.

In [0]:
## Q12 [ADVANCED] — SCD Types 3, 4, and 6 (Beyond Type 2)

Your product dimension tracks the **manager** of each sales rep. Business requirements:

* Analysts must see the **current** manager instantly (95% of queries).
* A quarterly report needs the manager history **one quarter back** — not the full history.
* Full audit history must still exist somewhere.

Compare **SCD Type 3** (add `previous_value` columns), **Type 4** (mini-dimension / history table), and **Type 6** (hybrid: 1+2+3 — current flag + effective dates + previous pointer), then design the tables for this requirement and write the MERGE-style logic for Type 3 (when a rep's manager changes: `current_manager` → `previous_manager`, write the new one). State the trade-offs of each.

**Interview points:** Almost everyone can recite Type 1 and Type 2; Types 3/4/6 are where you prove depth. The killer detail: Type 3 only keeps a **fixed window of history** (n columns = n periods back) — perfect here, terrible for full audit.

In [0]:
# Solution Q12 — SCD Type 3 (previous-column) design + change logic, and 4/6 comparison
from pyspark.sql import functions as F

print("""
THE THREE DESIGNS (trade-offs first — interviewers want this up front)
------------------------------------------------------------------------
SCD Type 3 — add previous_value columns to the SAME row:
  rep_dim(rep_id, rep_name, current_manager, previous_manager, manager_changed_on)
  + cheapest queries (no joins), fixed one-period lookback, NO audit beyond that.
  -> fits our requirement: 95% current + 1 quarter back; full audit delegated elsewhere.
SCD Type 4 — mini-dimension/history table:
  rep_dim(rep_id, ...current attrs...) + rep_manager_history(rep_id, manager, eff_from, eff_to)
  + unbounded history, current dims stay narrow and hot; queries need a join for history.
  -> the 'somewhere' where the full audit lives in our design.
SCD Type 6 — hybrid (Type 1+2+3 together):
  keep current_flag, effective_from/to AND previous_manager on each version row;
  views give current-only (flag) or history (all rows) or one-back (self-join).
  + most flexible, most storage + most complexity; worth it only when all access patterns exist.
""")

# --- Type 3 implementation: manager change moves current -> previous ---
reps = [(1, "Asha", "Ravi"), (2, "Ben", "Ravi"), (3, "Chen", "Meera")]
rep_dim = spark.createDataFrame(reps, ["rep_id", "rep_name", "current_manager"]) \
    .withColumn("previous_manager", F.lit(None).cast("string")) \
    .withColumn("manager_changed_on", F.lit(None).cast("date"))
spark.sql("DROP TABLE IF EXISTS day08_demo.rep_dim")
rep_dim.write.format("delta").saveAsTable("day08_demo.rep_dim")

# Source event: Asha's manager changes from Ravi to Priya on 2026-04-01
changes = [(1, "Priya")]
changes_df = spark.createDataFrame(changes, ["rep_id", "new_manager"]) \
    .withColumn("change_date", F.to_date(F.lit("2026-04-01")))
changes_df.createOrReplaceTempView("manager_changes")

spark.sql("""
MERGE INTO day08_demo.rep_dim d
USING manager_changes c ON d.rep_id = c.rep_id
WHEN MATCHED THEN UPDATE SET
  d.previous_manager = d.current_manager,   -- old value slides into the history column
  d.current_manager  = c.new_manager,
  d.manager_changed_on = c.change_date
""")
spark.sql("SELECT * FROM day08_demo.rep_dim ORDER BY rep_id").show()

row_1 = spark.sql("SELECT current_manager, previous_manager FROM day08_demo.rep_dim WHERE rep_id = 1").first()
assert row_1["current_manager"] == "Priya" and row_1["previous_manager"] == "Ravi"
print("Type 3 verified: current=Priya, previous=Ravi — current queries stay join-free, one lookback works.")

# Type 6 note to say out loud: a Type 6 version of this MERGE also inserts a new effective-dated
# row with is_current=true and flips the old row to is_current=false — Type 3's 'update in place'
# becomes 'update + insert'. Storage doubles per change; you buy unbounded history + fast current.
# Type 4 note: the same MERGE would instead append to rep_manager_history (eff_to = change_date
# for the old row, insert new row) and update rep_dim in place — history table grows, dim stays hot and narrow.

## Q13 [ADVANCED] — Late-Arriving Facts vs Late-Arriving Dimensions

Two incidents hit your warehouse on the same morning:

* **Incident A:** A fact row for `2026-03-31` arrives on April 5 with a `product_id` whose **dimension row landed weeks later** (the product launch data feed lagged).
* **Incident B:** A **dimension attribute update** (product moves from category `Electronics` to `Home`) arrives 3 days **after** the facts referencing that product were loaded.

Explain the difference between a **late-arriving fact** and a **late-arriving dimension** (sometimes called late-arriving dimension *attribute*), the damage each causes (orphan facts / wrong historical rollups), and the standard remedies: placeholder/unknown-member rows, inferred dimensions (natural-key stubs completed on arrival), and restatement vs as-of reporting for the late attribute. Show stub-then-complete logic for the inferred member.

**Interview points:** This is the question that separates warehouse people from people who memorized SCD types. The crisp framing: *late fact = the row itself is late (grain intact, history is just missing until it arrives); late dimension = the descriptive context for already-loaded facts is late (facts silently land with wrong/missing context)*.

In [0]:
# Solution Q13 — inferred dimension members + late fact/dimension handling
from pyspark.sql import functions as F

print("""
LATE-ARRIVING FACT
------------------
The fact row arrives after its date. Damage: history is incomplete until it arrives — historical
revenue for 2026-03-31 understated between Mar 31 and Apr 5. Grain is fine; nothing is 'wrong',
just missing.
Remedies:
  - Restate: append the late fact and rebuild/recompute affected aggregates (daily revenue for
    2026-03-31). Cost: downstream cubes/exports must refresh for a PAST period — plan backfill
    jobs that reprocess only affected partitions (WHERE order_date = '2026-03-31').
  - Where possible, correct at the source: the Kafka/ledger event date is the partition key, not
    arrival time.

LATE-ARRIVING DIMENSION (the sneaky one)
----------------------------------------
Facts reference a dimension key that does not exist YET (or the attribute update arrives after
the facts). Damage: facts join to an unknown/placeholder member -> rollups by category silently
wrong, or product revenue parked under '-99 Unknown'.
Remedies:
  - Unknown/placeholder member: every dim has key -99 'Unknown' — late-referenced keys land there
    instead of orphaning (classic Kimball).
  - Inferred dimension member: insert a STUB row keyed by the natural key (product_sku) with a
    generated surrogate key + is_inferred=true; when the real dim feed arrives, COMPLETE the
    stub in place (Type 1 style) — the facts now join correctly with NO restatement.
  - Late ATTRIBUTE change (Electronics -> Home): decide as-of vs restated — SCD2 gives as-of
    (facts keep the category true at sale time); a Type 1 restatement re-buckets history to the
    new value. Business must choose; never silently pick one.
""")

# --- Inferred-member demo: stub row on late fact, completed when the dim feed arrives ---
late_fact = [(9001, "SKU-TESLA-3", 45000.0)]
fact_df = spark.createDataFrame(late_fact, ["order_id", "product_sku", "amount"])

# Step 1: dim has no such member yet -> insert an inferred stub with a surrogate key
dim_feed_later = [("SKU-TESLA-3", "Model 3", "Electronics")]  # arrives 2 weeks later

spark.sql("DROP TABLE IF EXISTS day08_demo.product_dim")
spark.createDataFrame([(1, "SKU-X", "Widget X", "Home", False)],
        ["product_sk", "product_sku", "product_name", "category", "is_inferred"]) \
    .write.format("delta").saveAsTable("day08_demo.product_dim")

fact_df.createOrReplaceTempView("late_fact")
spark.sql("""
MERGE INTO day08_demo.product_dim d
USING (SELECT DISTINCT product_sku FROM late_fact) f
ON d.product_sku = f.product_sku
WHEN NOT MATCHED THEN  -- inferred member: stub with generated key, flagged
  INSERT (product_sk, product_sku, product_name, category, is_inferred)
  VALUES ((SELECT COALESCE(MAX(product_sk),0)+1 FROM day08_demo.product_dim),
          f.product_sku, NULL, 'Unknown', true)
""")

# Step 2: real dimension feed arrives later -> complete the stub in place (keep the key!)
spark.createDataFrame(dim_feed_later, ["product_sku", "product_name", "category"]) \
    .createOrReplaceTempView("dim_feed")
spark.sql("""
MERGE INTO day08_demo.product_dim d
USING dim_feed f ON d.product_sku = f.product_sku
WHEN MATCHED THEN UPDATE SET
  d.product_name = f.product_name,
  d.category     = f.category,
  d.is_inferred  = false       -- stub becomes a real member; surrogate key UNCHANGED
""")

result = spark.sql("SELECT * FROM day08_demo.product_dim WHERE product_sku = 'SKU-TESLA-3'").first()
assert result["category"] == "Electronics" and result["is_inferred"] is False and result["product_sk"] == 2
spark.sql("SELECT * FROM day08_demo.product_dim ORDER BY product_sk").show(truncate=False)

print("Verified: the late fact's SKU now has a complete dimension row under the SAME surrogate key")
print("-> no fact restatement needed — that is the entire value of inferred members.")

## Q14 [INTERMEDIATE] — Snowflake Zero-Copy Cloning

"Your dev team wants a full copy of a 5 TB production schema for testing this week's migration. Naive `CREATE TABLE ... AS SELECT` would take hours and double storage. Explain Snowflake **zero-copy cloning**: how `CREATE DATABASE dev CLONE prod` works at the micro-partition level, what is and isn't copied, what happens on first write to the clone, and the traps (time-travel interaction, cloning a schema vs a database, permission copying)."

**Interview points:** Zero-copy cloning is a top Snowflake differentiator asked at every company that runs it. The core sentence to nail: *the clone shares the underlying micro-partition files — no data is copied at clone time; copy-on-write gives the clone its own partitions only as it diverges.*

In [0]:
# Solution Q14 — zero-copy cloning (explanatory with SQL)

print("""
HOW ZERO-COPY CLONING WORKS
----------------------------
CREATE DATABASE dev CLONE prod;   -- or CLONE at schema/table level

Micro-partition level: Snowflake does NOT copy any micro-partition files. It clones the METADATA:
the new object's catalog entries point at the SAME micro-partitions as the source. A 5 TB schema
clones in seconds regardless of size, with ~zero immediate storage cost.

Copy-on-write divergence:
- WRITE to the clone (UPDATE/DELETE/INSERT): affected micro-partitions are re-written for the
  CLONE only (a new set of files replaces its pointers); the source is untouched.
- WRITE to the SOURCE after cloning: same in reverse — the clone's view is frozen at the clone
  point; the source's changes create new partitions for the source.
- Net: storage grows only by the DIVerged data, never the full copy.

What is cloned / NOT cloned
----------------------------
Cloned: tables (data + structure), schemas/databases structure, file format references.
NOT cloned (the traps that make this an interview question):
- GRANTS: by default the clone does NOT inherit the source's object-level grants (cloning a
  database/schema can carry grants with COPY GRANTS, but a TABLE clone needs explicit re-grant).
- Stages (internal ones), pipes/tasks, streams, sequences' state, policies tagged to objects,
  resource monitors, and anything referencing objects OUTSIDE the cloned boundary.
- Future grants: re-apply. Always script permissions as code, never click.

Time-travel interaction
-----------------------
The clone captures the source at its CURRENT version. Time travel on the clone goes back to
CLONE time, not to the source's earlier history. And cloning does NOT reset the source's
time-travel/Fail-safe retention.

TYPICAL USES
-------------
-- Dev/test from prod, instantly:
CREATE SCHEMA staging CLONE prod.analytics;
-- Pre-migration snapshot / instant backup before a risky change:
CREATE TABLE orders_backup CLONE orders;
-- Monthly snapshot for reporting freezing (no storage cost until divergence)

OPERATIONAL NOTES
------------------
- Drop clones when done — diverged data still costs storage.
- A clone is NOT a backup: accidental DROP of both source and clone can still lose data,
  and Fail-safe windows still apply.
- On Databricks Delta the comparable primitive is 'shallow clone' (CREATE TABLE ... SHALLOW CLONE),
  same metadata-copy idea on Delta files — nice cross-platform point to mention.
""")

## Q15 [ADVANCED] — Snowflake Search Optimization Service

"Your dashboard runs a `SELECT ... WHERE customer_email = 'x@y.com'` lookup on a 2-billion-row table. It filters 200B+ micro-partitions down to a few and takes 45+ seconds — pruning by the clustered key doesn't help because the lookup column isn't part of the clustering. What is Snowflake's **Search Optimization Service (SOS)**, how does it work (search access path), what does it cost, and when is it the right/wrong answer vs clustering keys vs a materialized view?"

**Interview points:** This tests if you know Snowflake's answer to *point-lookup on huge tables* — and the cost model discipline behind it: SOS is billed by **storage of the index + maintenance compute on every DML**, so it's only right for big tables with selective, infrequently-updated lookups.

In [0]:
# Solution Q15 — Search Optimization Service (explanatory with SQL)

print("""
WHAT SOS IS
-----------
A managed, table-level index infrastructure (like an enterprise search index under the hood).
Enabled once per table (costs credits) and then per-column:

  ALTER TABLE orders ADD SEARCH OPTIMIZATION;              -- table-level (point lookup method)
  ALTER TABLE orders ADD SEARCH OPTIMIZATION ON EQUALITY(customer_email);
  ALTER TABLE orders ADD SEARCH OPTIMIZATION ON SUBSTRING(product_desc);  -- substring/like

HOW IT WORKS (search access path)
----------------------------------
Snowflake builds a 'search access path' — a partition-to-value index stored separately from the
table. At query time the optimizer can, for an equality/substring predicate, look up WHICH
micro-partitions could contain the value and prune to only those — skipping the scan of
hundreds of thousands of partitions. Because it is maintained by Snowflake, there is no index
syntax in your queries — the optimizer just uses it when a qualifying predicate appears.
Typical wins: point lookups on multi-billion-row tables going from tens of seconds to
sub-second; also LIKE with leading wildcard via the substring method.

COST MODEL (where interviews are won)
--------------------------------------
1. BUILD cost: one-time compute to index existing data (billed per TB processed).
2. STORAGE cost: the index itself (can be a meaningful % of table size — assume ~10-25%).
3. MAINTENANCE cost: EVERY insert/update/delete on the table updates the index — background
   serverless compute. On a table with constant heavy DML, maintenance can exceed the benefit.

DECISION RULES (say these explicitly)
-------------------------------------
- RIGHT: very large table, selective equality lookups on non-clustering columns, modest DML.
- WRONG choice when:
  * The predicate is a RANGE (date > X) — SOS equality doesn't help; clustering keys do.
  * The column is already the clustering key — natural pruning already works.
  * The table is small (millions of rows) — full scans are already fast; you pay for nothing.
  * Heavy, continuous DML — maintenance cost swamps savings; consider a materialized view of
    the lookup-relevant columns instead.
- ALTERNATIVES to compare: clustering keys (range/group pruning), materialized views (frequent
  fixed-shape lookups, extra storage + refresh), external search index (only if you need text
  search semantics).

GOVERNANCE
----------
  SHOW TABLES LIKE 'orders';
  ALTER TABLE orders ADD SEARCH OPTIMIZATION ON EQUALITY(customer_email);
  -- verify + measure:
  SELECT * FROM TABLE(INFORMATION_SCHEMA.SEARCH_OPTIMIZATION_TABLE_FUN(...)); -- usage view
  DESCRIBE SEARCH OPTIMIZATION ON orders;   -- what methods/columns are indexed
Always A/B test on a cloned environment (zero-copy clone -> enable SOS -> compare) before
paying for it in production.
""")

In [0]:
## Q16 [ADVANCED] — Monitoring Pipelines with System Tables (system.lakeflow.*)

"Your manager asks: 'Give me a self-serve view of every scheduled pipeline run in the last 7 days — success rate, duration trend, and failures by error class — without touching the pipelines.' What Databricks **system tables** make this possible? Write the query skeleton for a weekly reliability scorecard and explain what each table gives you."

**Interview points:** System tables are Databricks' answer to 'monitor the platform with the platform.' Knowing that `system.lakeflow.job_run_timeline` (and `system.lakeflow.table_info`, `system.access.*` etc.) exist — and that they're queryable SQL — instantly signals production experience.

In [0]:
# Solution Q16 — system tables for pipeline observability (query skeletons)

print("""
THE RELEVANT SYSTEM TABLES (schema: system.lakeflow.*, system.billing.*, system.access.*)
------------------------------------------------------------------------------------------
system.lakeflow.job_run_timeline     -- one row per job/pipeline TASK run: state (PENDING,
                                       RUNNING, SUCCEEDED, FAILED, SKIPPED, CANCELLED,
                                       TIMEDOUT), start/end times, run duration, job_id,
                                       task run id, repair history. THE backbone of reliability
                                       metrics. Retention: ~365 days.
system.lakeflow.job_timeline          -- job-level lifecycle (created, updated, deleted, task
                                       changes) — what exists and how it changed.
system.billing.usage                 -- every consumed DBU with tags/cluster info — join to runs
                                       to get cost per run/pipeline (FinOps).
system.billing.list_prices           -- pricing to convert DBUs to $.
system.access.audit                  -- who did what (includes job runs triggered by whom).
system.lakeflow.table_info           -- table lineage/metadata inventory for data-level checks.

This is pipeline-level monitoring as pure SQL — no agents, no extra tool to buy.
""")

# Weekly reliability scorecard skeleton (works as-is if you have a workspace with system tables):
scorecard_sql = """
WITH runs AS (
  SELECT job_id,
         result_state,
         period_start_time,
         (end_time - start_time) AS duration
  FROM system.lakeflow.job_run_timeline
  WHERE period_start_time >= current_timestamp() - INTERVAL 7 DAYS
    AND result_state IN ('SUCCEEDED', 'FAILED', 'TIMEDOUT')  -- exclude running/pending
)
SELECT
  job_id,
  COUNT(*)                                        AS total_finished_runs,
  ROUND(100 * SUM(CASE WHEN result_state = 'SUCCEEDED' THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                  AS success_rate_pct,
  ROUND(AVG(duration) / 60, 1)                    AS avg_run_minutes,
  MAX(duration) / 60                              AS max_run_minutes,
  SUM(CASE WHEN result_state IN ('FAILED','TIMEDOUT') THEN 1 ELSE 0 END) AS failures
FROM runs
GROUP BY job_id
HAVING COUNT(*) >= 1
ORDER BY success_rate_pct ASC, failures DESC
"""
print("Scorecard query skeleton:")
print(scorecard_sql)

# Try it live if the workspace exposes system tables; degrade gracefully otherwise.
try:
    df = spark.sql(scorecard_sql).limit(5)
    print("Live sample (top 5 least reliable jobs this week):")
    df.show(truncate=False)
except Exception as e:
    print(f"(Live system-table query not available here: {str(e)[:120]}... — skeleton is the point.)")

# Interview follow-ups to be ready for:
# - Data-level vs pipeline-level: system tables tell you the JOB failed; data freshness/quality
#   metrics (Q18) tell you the OUTPUT is wrong even when the job is green.
# - Cost per run: join job_run_timeline (run_key) to billing.usage (usage_metadata.job_run_id)
#   -> $/successful run per pipeline; the FinOps follow-up is guaranteed.
# - Alerting: point a Databricks SQL alert at this exact query — SLA breach when
#   success_rate_pct < 95 or avg_run_minutes grows 50% WoW.

In [0]:
## Q17 [ADVANCED] — The 3 AM Failure Walkthrough: Debug a Dead Pipeline Methodically

"It's 3 AM. Your nightly silver-layer pipeline failed at 2:47 AM after running fine for 3 weeks. The on-call alert just says 'Task 4: Silver Orders — FAILED.' Walk me through your debugging process, end to end, using Databricks tooling. Structure your answer and justify each step."

Cover: alert triage (is it the code, the data, the platform?), run history and Repair Runs, driver logs vs stdout/stderr, log4j levels and Python logging vs print, Spark UI / query profile for the failing stage, `DESCRIBE HISTORY` on the output table, and how you decide between rerun vs rollback vs hotfix.

**Interview points:** Interviewers grade *process*, not luck. The strongest answers are a repeatable triage order — from cheapest evidence to most expensive — and end with 'and then I make sure this never wakes anyone again' (alert fix, runbook entry).

In [0]:
# Solution Q17 — the 3 AM debugging walkthrough (explanatory, staged)

print("""
STAGE 0 — TRIAGE IN 60 SECONDS (cheapest evidence first)
---------------------------------------------------------
1. Which failure class is it? Open the job run page: result_state + failure message.
   - 'Task failed with a Python exception' -> code/data problem.
   - Cluster/node lost, cloud provider error, spot reclaim -> platform problem.
   - 'Job timed out' -> data volume or a hang (backpressure/deadlock).
2. Did upstream tasks SUCCEED? If task 3 (bronze) is also red, debug THAT — don't stare at task 4.

STAGE 1 — WHAT CHANGED? (the 3-weeks-green clue)
------------------------------------------------
A pipeline that ran 3 weeks and broke tonight almost never has NEW code — it has NEW DATA.
- Was the job definition changed? Run history -> compare the failing run's config to last week's.
- What was different about today's input? Usually: volume spike (Black Friday, backfill, a
  partner dumping duplicates) or a schema change from an upstream source system.

STAGE 2 — READ THE ACTUAL ERROR (logs)
--------------------------------------
- Driver logs (Cluster -> Driver logs / Stderr): the Python traceback / JVM exception with
  the line number. This is the single most information-dense artifact.
- My own pipeline's log discipline pays off here: structured Python logging (logger =
  logging.getLogger('silver_orders')) with levels, NOT print(): prints bypass levels, can't be
  filtered, and aren't routed consistently. log4j levels (ERROR/WARN/INFO/DEBUG) map to Spark's
  internals — DEBUG only when reproducing, never in prod volume.
- What to have logged at INFO: batch_id, row counts in/out, watermark, schema, and stage
  boundaries. At 3 AM you want COUNTS, not vibes.

STAGE 3 — THE FAILING QUERY (Spark UI / query profile)
-------------------------------------------------------
- Spark UI of the failed run (preserved for the job): which stage failed, task-level error,
  shuffle spill, GC time, stragglers.
- Query Profile (SQL): the operator-level plan; a sudden shuffle explosion is a cardinality
  change (duplicate join keys!) — the classic 'worked for 3 weeks then data doubled' signature.

STAGE 4 — THE OUTPUT TABLE (DESCRIBE HISTORY)
----------------------------------------------
- DESCRAIBE... i.e. DESCRIBE HISTORY ON silver_orders ORDER BY version DESC: did the failing run
  WRITE anything before dying? A partial write means the output is in an unknown state:
  - If the job is idempotent + writes are transactional (Delta MERGE / replaceWhere): just fix
    and RERUN (Repair Run re-runs only the failed task and its dependents — not the whole DAG).
  - If non-idempotent partial writes: RESTORE the table to the last good version, then rerun.

STAGE 5 — DECIDE: RERUN / ROLLBACK / HOTFIX
--------------------------------------------
- Transient (cloud hiccup, spot loss): Repair Run now. Zero code changes.
- Bad data (schema drift, duplicates): fix forward — quarantine the poison batch (WHERE clause
  or DLQ), rerun. Do NOT try to hand-edit the table.
- Bad code deployed recently: RESTORE output tables to pre-failure versions and roll back the
  deployment.

STAGE 6 — 3:47 AM AND THE LAST STEP EVERYONE FORGETS
------------------------------------------------------
- Update the alert so THIS failure pages with a useful message (row-count anomaly, not just
  'FAILED').
- One-paragraph runbook entry / incident ticket: symptom, root cause, fix. Next on-call thanks you.
- If it was data volume: add a volume anomaly check so it warns BEFORE it fails next time.
""")

# Quick reference — the three tools and what each is for:
print("""
CHEAT SHEET
-----------
Driver log / stderr : the actual exception + line number          -> WHAT broke
Spark UI / Profile  : stage, tasks, shuffle, GC, stragglers        -> WHERE and WHY it broke
DESCRIBE HISTORY   : table versions, partial writes, who/when     -> is the OUTPUT safe
Repair Run         : rerun from the failed task onwards           -> cheapest recovery
RESTORE             : table back to last good version               -> rollback of last resort
""")

## Q18 [ADVANCED] — Lakehouse Monitoring: Data-Level Quality & Freshness Metrics

"You monitor pipelines (jobs run green), but twice this quarter a job SUCCEEDED and the dashboard underneath showed stale/wrong data for hours. Design **data-level monitoring** for a critical table `gold.orders_daily` using Databricks **Lakehouse Monitoring**: what metrics it computes (freshness, volume, row count, nullness, column drift), how a monitor is attached to a table, and how SQL alerts consume the metrics tables it produces. Explain the pipeline-level vs data-level monitoring distinction."

**Interview points:** The trap is only ever monitoring the *job*. Lakehouse Monitoring is the 2024+ Databricks answer to 'the job was green but the data was wrong' — knowing the metrics-table pattern (and that it produces actual queryable tables) is a strong differentiator.

In [0]:
# Solution Q18 — Lakehouse Monitoring + the metrics-table pattern

print("""
PIPELINE-LEVEL vs DATA-LEVEL (say this distinction FIRST)
----------------------------------------------------------
Pipeline-level: did the JOB succeed? Job runs, duration, retries (system tables, Jobs UI).
Data-level: is the OUTPUT correct, fresh, and complete? A job can be green while writing
0 rows, yesterday's data, or 40% nulls in a business column — pipeline monitoring never sees it.
Mature platforms have both; data-level catches the failures users actually notice.

WHAT LAKEHOUSE MONITORING IS
-----------------------------
A Databricks feature (Delta Table Monitor, e.g. via DatabricksMonitor resource / SDK / UI) that
runs scheduled metric COMPUTATION over your table and persists results as Delta metrics tables
you can query like any table — no custom logging framework to build.

It computes per time-window slices (and per monitored column):
- Row count / volume anomalies             -> the '0 rows loaded' failure
- Data freshness (max timestamp lag)        -> the 'stale all weekend' failure
- Null counts/rates and type anomalies       -> the '40% nulls in customer_id' failure
- Basic statistical profile + drift over time (mean/min/max, distribution shift) -> silent
  upstream semantic changes
Two modes: snapshot monitors (profile at points in time) and time-series monitors (slice by an
event-time column — usually the right choice for fact tables).

ATTACHING A MONITOR (conceptual — resource/API form)
-------------------------------------------------------
from databricks.sdk.service.catalog import MonitorTimeSeries
mon = SdkMonitors.create_monitor(
  table_name = 'gold.orders_daily',
  time_series = MonitorTimeSeries(
     timestamp_col = 'order_date',
     granularities = ['1 day'],
     output_schema  = 'monitoring'),
  assets_dir = 's3://.../monitors/orders_daily')
# The monitor materializes tables: monitoring.orders_daily_profile, ..._drift_metrics, etc.

THE METRICS-TABLE PATTERN (the second half of the answer)
----------------------------------------------------------
Because the metrics are DELTA TABLES, alerts are just SQL over them:

-- Freshness SLA: alert when today's slice is missing or max(order_date) is > 1 day behind
SELECT 1 AS breach
FROM monitoring.orders_daily_profile
WHERE slice_timestamp = current_date()
  AND row_count < 1000;          -- volume floor for the day

-- Drift: null-rate on a business column jumps vs 7-day baseline
SELECT 1 AS breach
FROM monitoring.orders_daily_null_metrics
WHERE col_name = 'customer_id' AND null_rate > 0.05;

Wire each into a Databricks SQL Alert (schedule + threshold + notification channel/Slack)
and you have the full loop: table -> monitor -> metrics table -> SQL alert -> page a human.
""")

# Demonstrate the same metrics computed by hand (what a monitor automates) on a tiny table:
from pyspark.sql import functions as F

orders = spark.createDataFrame([
    ("2026-01-01", "o1", "c1"), ("2026-01-01", "o2", None),
    ("2026-01-02", "o3", "c2"),
], ["order_date", "order_id", "customer_id"])
orders.createOrReplaceTempView("gold_orders")

quality_sql = """
SELECT
  MAX(order_date)                                        AS freshest_date,
  DATEDIFF(CURRENT_DATE(), MAX(to_date(order_date)))     AS freshness_lag_days,
  COUNT(*)                                               AS total_rows,
  ROUND(AVG(CASE WHEN customer_id IS NULL THEN 1.0 ELSE 0.0 END), 2) AS null_rate_customer_id
FROM gold_orders
"""
m = spark.sql(quality_sql).first()
print("Hand-computed monitor metrics (what Lakehouse Monitoring automates):")
spark.sql(quality_sql).show()
assert float(m["null_rate_customer_id"]) == round(1/3, 2)  # 1 null of 3 rows
print(f"\nFreshness lag = {m['freshness_lag_days']} days, null rate = {m['null_rate_customer_id']} ")
print("-> point SQL alerts exactly like these at the monitor's metrics tables in production.")

In [0]:
# Scorecard — Day 08

Rate yourself per question (1 = couldn't solve, 2 = solved with hints, 3 = solved cleanly, 4 = solved cleanly AND explained trade-offs out loud):

| Q | Topic | Difficulty | Self-Rating (1-4) | Notes |
|---|-------|------------|-------------------|-------|
| Q1 | SQL — gaps-and-islands sessionization | INTERMEDIATE | | |
| Q2 | SQL — cohort analysis retention | ADVANCED | | |
| Q3 | Python — LRU cache (OrderedDict + linked list) | INTERMEDIATE | | |
| Q4 | Python — BFS shortest path in a graph | ADVANCED | | |
| Q5 | PySpark — CDC change-log processing | INTERMEDIATE | | |
| Q6 | PySpark — skewed join salting | ADVANCED | | |
| Q7 | Spark Opt — AQE internals | ADVANCED | | |
| Q8 | Spark Opt — speculative execution | ADVANCED | | |
| Q9 | ETL — exactly-once semantics | ADVANCED | | |
| Q10 | ETL — watermarks & late data | ADVANCED | | |
| Q11 | ETL — Delta OPTIMIZE / ZORDER / VACUUM | INTERMEDIATE | | |
| Q12 | Data Modeling — SCD Types 3/4/6 | ADVANCED | | |
| Q13 | Data Modeling — late-arriving facts vs dims | ADVANCED | | |
| Q14 | Snowflake — zero-copy cloning | INTERMEDIATE | | |
| Q15 | Snowflake — search optimization service | ADVANCED | | |
| Q16 | Monitoring — system tables scorecard | ADVANCED | | |
| Q17 | Monitoring — 3 AM failure debugging walkthrough | ADVANCED | | |
| Q18 | Monitoring — Lakehouse Monitoring metrics | ADVANCED | | |

**Week 2 target:** every ADVANCED question at 3+, every INTERMEDIATE at 4.

**Tomorrow (Day 09):** more multi-step ADVANCED scenarios — recursive CTEs / time-series gaps in SQL, generators & decorators in Python, MERGE INTO SCD in PySpark, shuffle & AQE tuning, backpressure & circuit breakers in ETL, Data Vault, Snowflake resource monitors & policies, orchestration at scale.